<h1 style="text-align:center;">37th International Summer School of the Swiss Association of Actuaries on</h1>
<h4 style="text-align:center;">“Insurance: Innovations for Products, Sustainability and Regulation”,</h4>
<p style="text-align:center;">Lausanne, 10.-14.08.2026</p>

# Day 4

## Overview

1) **Pay-out plans**\
Come up with some examples for pay-out plans and compare them to the Merton (1969)version with exponentially distributed remaining life time both by their theoretical con-cepts and by simulated examples.

2) **Worst case and dynamic reinsurance**\
Show that the solution of the dynamic reinsurance worst-case problem given in Theorem1 of Talk 5 has exactly this form.

3) **Climate scenarios (Bonus)**\
Perform an internet search for current dynamic models for the evolution of the average temperature increase in the last ten years.

In [ ]:
import numpy as np
import pandas as pd

from utils.common import *
from utils.simulators import *
from utils.viz import *

## Exercise 1

Come up with some examples for pay-out plans and compare them to the Merton (1969)version with exponentially distributed remaining life time both by their theoretical con-cepts and by simulated examples.

### Payout plans under an uncertain remaining lifetime

Consider a Black-Scholes and an investor who wants to maximize

$$\mathbb{E}\left[\int_0^\infty \mathrm{e}^{-\delta t}\log(c_t)\,\mathrm{d}t\right] .$$

Suppose that the fund manager invests a constant fraction $\pi$ of wealth in the risky asset and applies a deterministic payout factor $c_t=\phi(t)X_t$. Then the investor's wealth satisfies

$$\mathrm{d}X^\phi_t = X^\phi_t\left[ r + \pi(\mu-r) - \phi(t) \right] \mathrm{d}t + \pi\sigma X^\phi_t \,\mathrm{d}W_t.$$

This is a geometric SDE. Applying Itô's formula to $\log X^\phi_t$ gives

$$\mathrm{d}\log X^\phi_t = \left[ r + \pi(\mu-r)-\phi(t)-\frac{1}{2}(\pi\sigma)^2 \right] \mathrm{d}t + \pi\sigma \,\mathrm{d}W_t.$$

Integrating from $0$ to $t$ yields

$$\log\left( \frac{X^\phi_t}{X^\phi_0} \right) = \int_0^t\left[ r + \pi(\mu-r) - \phi(s) - \frac{1}{2}(\pi\sigma)^2 \right] \mathrm{d}s + \pi\sigma W_t.$$

Consequently,

$$X^\phi_t = X^\phi_0\exp\left(\int_0^t\left[ r + \pi(\mu-r) - \phi(s) - \frac{1}{2}(\pi\sigma)^2\right] \mathrm{d}s + \pi\sigma W_t \right).$$

Defining the market price of risk by

$$\theta=\frac{\mu-r}{\sigma},$$

and letting

$$X^\pi_t = \exp\left(\left(r + \pi(\mu-r) - \frac{1}{2}(\pi\sigma)^2 \right)t + \pi\sigma W_t\right),$$

we can write

$$X^\phi_t = X^\pi_t \cdot X^\phi_0 \exp\left(-\int_0^t\phi(s)\,\mathrm{d}s\right).$$

In the standard Merton problem, the optimal risky fraction $\pi^\ast$ and optimal consumption rate factor $\phi^\ast$ are given by

$$\pi^\ast=\frac{\mu-r}{\sigma^2} \quad\text{and}\quad \phi^\ast=\delta.$$

Now, let $\tau\sim\mathrm{Exp}(\lambda)$ be independent of the stock market. Its survival function is $\mathbb{P}(\tau>t)=\mathrm{e}^{-\lambda t}$ and $\mathbb{E}[\tau]=\frac{1}{\lambda}$. For log utility,

$$\mathbb{E}\left[\int_0^\tau \mathrm{e}^{-\delta t}\log(c_t)\,\mathrm{d}t\right]=\mathbb{E}\left[\int_0^\infty \mathrm{e}^{-(\delta+\lambda)t}\log(c_t)\,\mathrm{d}t\right].$$

Thus mortality acts like additional discounting, and our optimal consumption rate factor is $\phi^\ast=\delta+\lambda$. To see how this optimal payout plan under the assumption of an exponential lifetime compares to other payout plans, we consider the following four cases:

1. **Standard infinite horizon:** $c_t = \delta X_t$.
2. **Exponential lifetime:** $c^\ast_t = (\delta + \lambda)X_t$.
3. **Fixed horizon:** Choose $H = \mathbb{E}[\tau]$ and $c_t = \frac{X_t}{\mathcal{E}^{-\delta}(H-t)}$.
4. **Quantile horizon:** Choose $T_z$ from $\mathbb{P}(\tau>T_z)=z$, hence $T_z=-\frac{\log z}{\lambda}$, and take $c_t = \frac{X_t}{\mathcal{E}^{-\delta}(T_z-t)}$.

Note that $c_t$ as chosen in cases 3 and 4 are the optimal payout processes, if one were to live exactly until the fixed horizon. Since one may live shorter (or longer), these plans may lead to a depletion of the fund before death (in case 4 with a probability of exactly $z$) but, in turn, allow for higher consumption rates while alive.

In [ ]:
def simulate_payout_plans(initial_wealth: float, expected_lifetime: float, discount_rate: float, tail_probability: float, market_params: FinancialMarket, sde_params: SDESimulationParameters) -> SDEOutput:
    '''Simulate four payout plans under common market paths and lifetime draws.'''

    if expected_lifetime <= 0:
        raise ValueError("Expected lifetime must be positive.")
    if discount_rate < 0:
        raise ValueError("The discount rate must be non-negative.")
    if not 0 < tail_probability < 1:
        raise ValueError("The tail probability must lie strictly between zero and one.")

    mortality_rate   = 1 / expected_lifetime
    quantile_horizon = -np.log(tail_probability) / mortality_rate

    if sde_params.time_horizon < quantile_horizon:
        raise ValueError("The simulation horizon must not be shorter than the quantile horizon.")

    out = simulate_brownian_path(params=sde_params)
    t = out.time_grid
    dt = t[1] - t[0]
    W = out.paths["W"]

    # Consumptionless fund wealth under the growth-optimal strategy (X^\phi_0 * X^{\pi^\ast}_t)
    pi_star = ...
    fund = ...

    # Infinite-horizon Merton plans
    # Case 1: Standard infinite horizon (no mortality)
    standard_wealth      = ...
    standard_consumption = discount_rate * standard_wealth

    # Case 2: Exponential lifetime (mortality-adjusted, this is the optimal plan under uncertain exp~lifetime)
    mortality_wealth      = ...
    mortality_consumption = (discount_rate + mortality_rate) * mortality_wealth

    # Case 3/4: Fixed horizon (Used for both mean and quantile horizons)
    # computes exp(-\int_0^t \phi(s) ds)
    def fixed_horizon_plan(horizon):
        remaining  = np.maximum(horizon - t, 0.0)
        active     = t <= horizon

        if np.isclose(discount_rate, 0.0):
            wealth_factor      = ...
            consumption_factor = ...
        else:
            wealth_factor      = ...
            consumption_factor = ...

        wealth      = fund * wealth_factor[:, np.newaxis]
        consumption = fund * consumption_factor[:, np.newaxis]

        wealth[~active, :]      = 0.0
        consumption[~active, :] = 0.0

        return wealth, consumption

    mean_horizon_wealth, mean_horizon_consumption = fixed_horizon_plan(expected_lifetime)
    quantile_wealth, quantile_consumption         = fixed_horizon_plan(quantile_horizon)

    # Draw lifetimes from an exponential distribution with the specified expected lifetime
    lifetime = np.random.exponential(scale=expected_lifetime, size=sde_params.num_paths)

    return SDEOutput(
        time_grid=t,
        paths={
            "standard_wealth": standard_wealth,
            "standard_consumption": standard_consumption,
            "standard_cumulative_consumption": cumulative_trapezoid(standard_consumption, dt),
            "mortality_wealth": mortality_wealth,
            "mortality_consumption": mortality_consumption,
            "mortality_cumulative_consumption": cumulative_trapezoid(mortality_consumption, dt),
            "mean_horizon_wealth": mean_horizon_wealth,
            "mean_horizon_consumption": mean_horizon_consumption,
            "mean_horizon_cumulative_consumption": cumulative_trapezoid(mean_horizon_consumption, dt),
            "quantile_wealth": quantile_wealth,
            "quantile_consumption": quantile_consumption,
            "quantile_cumulative_consumption": cumulative_trapezoid(quantile_consumption, dt),
            "lifetime": lifetime,
            "driver": W,
        },
    )

### Numerical validation

The simulation should reproduce the defining properties of the strategies. All funds start at $x$, the mortality-adjusted plan consumes the fraction $\delta+\lambda$ of wealth, and the quantile plan is exhausted at $T_z$. We also verify the definition $\mathbb{P}(\tau>T_z)=z$ analytically.

In [ ]:
x                 = 100_000.0
expected_lifetime = 15
discount_rate     = 0.025
tail_probability  = 0.1 # z chosen in the quantile-horizon plan such that P(tau > T_z) = z

mortality_rate   = 1 / expected_lifetime
quantile_horizon = -np.log(tail_probability) / mortality_rate

Tmax = 1.5*max(quantile_horizon, expected_lifetime) # We simulate a bit beyond the quantile horizon to see the full plan.

market = FinancialMarket(risk_free_rate=0.025, risk_premium=0.05, volatility=0.25)
params = SDESimulationParameters(time_horizon=Tmax, time_steps=int(52*Tmax), num_paths=10_000)

payout_plans   = simulate_payout_plans(x, expected_lifetime, discount_rate, tail_probability, market, params)
quantile_index = np.argmin(np.abs(payout_plans.time_grid - quantile_horizon))

lifetimes_larger_than_quantile = payout_plans.paths["lifetime"] > quantile_horizon

check_condition(
    "All plans start at the initial wealth",
    all(
        np.allclose(payout_plans.paths[f"{key}_wealth"][0, :], x)
        for key in ["standard", "mortality", "mean_horizon", "quantile"]
    ),
)
check_value(
    "Mortality-adjusted consumption-to-wealth rate",
    payout_plans.paths["mortality_consumption"][0, 0]
    / payout_plans.paths["mortality_wealth"][0, 0],
    discount_rate + mortality_rate,
)
check_value(
    "Survival probability at the quantile horizon",
    np.mean(lifetimes_larger_than_quantile),
    tail_probability,
    abs_tol=1e-3,
    rel_tol=1e-5
)
check_condition(
    "Quantile-horizon wealth is zero from the end of the plan",
    np.allclose(payout_plans.paths["quantile_wealth"][quantile_index + 1:, :], 0.0),
)

### Interactive comparison

The first two panels compare fund wealth and annual payouts on one common market path. The third panel combines average consumption received before death with the probability that a fixed-horizon plan ends while the individual is still alive. Here, a stylized actuarially fair annuity with annual rate $x(r+\lambda)$ (see remark) is included as a longevity-pooling reference.

#### Remark - Stylized actuarially fair annuity

Suppose an individual pays an initial premium $x$ and receives a constant payment rate $p$ until death. Let the remaining lifetime satisfy $\tau\sim\mathrm{Exp}(\lambda)$, so that

$$\mathbb{P}(\tau>t)=\mathrm{e}^{-\lambda t}.$$

At a constant risk-free interest rate $r$, the expected present value of the annuity payments is

$$\begin{aligned}
\mathbb{E}\left[\int_0^\tau \mathrm{e}^{-rt}p\,\mathrm{d}t\right]
&=\int_0^\infty \mathrm{e}^{-rt}p\,\mathbb{P}(\tau>t)\,\mathrm{d}t\\
&=p\int_0^\infty \mathrm{e}^{-(r+\lambda)t}\,\mathrm{d}t\\
&=\frac{p}{r+\lambda}.
\end{aligned}$$

For an actuarially fair annuity, the expected present value of the payments equals the initial premium:

$$x=\frac{p}{r+\lambda}.$$

Consequently, the constant actuarially fair payment rate is

$$\boxed{p=x(r+\lambda).}$$

The decomposition

$$p=xr+x\lambda$$

has a useful interpretation. The term $xr$ is the investment return on the initial capital, while $x\lambda$ is the mortality credit generated by pooling longevity risk: the remaining capital of individuals who die helps finance the continuing payments to survivors.

In [ ]:
explore_payout_plans(simulate_payout_plans)

### Interpretation

- Mortality raises the optimal initial payout rate from $\delta$ to $\delta+\lambda$: money is consumed faster because utility is obtained only while the individual is alive.
- A plan ending at $\mathbb{E}[\tau]$ is not a longevity hedge. For an exponential lifetime, $\mathbb{P}(\tau>\mathbb{E}[\tau])=\mathrm{e}^{-1}\approx36.8\%$.
- The quantile plan makes the longevity-risk tolerance explicit. By construction, only the fraction $z$ survives beyond $T_z$.
- No strategy dominates pathwise.

## Exercise 2

Show that the solution of the dynamic reinsurance worst-case problem given in Theorem1 of Talk 5 has exactly this form.

### Dynamic reinsurance and worst-case default

Before default, let $a_1(t)\in[0,1]$ denote the retained risk exposure; after default the exposure is $a_0$. We then consider strategies of the form
$$a(t) = \begin{cases} a_1(t), & t<\tau,\\ a_0, & t\geq\tau. \end{cases} ,$$
where $\tau$ is the default time. A larger $a$ means less reinsurance. The primary insurer loses

$$C + (1 - a_1(\tau))F$$

if the reinsurer defaults at time $\tau$. The robust objective is

$$\sup_{a\in\mathcal{A}}\inf_{\tau\in\Theta}\mathbb{E}\left[-\mathrm{e}^{-\gamma R^{a,\tau}(T)}\right].$$

### Step 1: Solve the problem after default

After default, the insurer uses the constant exposure $a_0$. Conditional on the information available at time $\tau < t$, terminal surplus is

$$R_T=R_t+\left[\mu-(1-a_0)\lambda\right](T-t)+a_0\sigma(W_T-W_t).$$

Since $W_T-W_t\sim\mathcal{N}(0,T-t)$,

$$\mathbb{E}\left[\mathrm{e}^{-\gamma a_0\sigma(W_T-W_t)}\mid\mathcal{F}_t\right]=\exp\left[\frac{\gamma^2a_0^2\sigma^2}{2}(T-t)\right].$$

Consequently,

$$\mathbb{E}\left[-\mathrm{e}^{-\gamma R_T}\mid\mathcal{F}_t\right]=-\mathrm{e}^{-\gamma R_t}\exp\left[\left(-\gamma\mu+q(a_0)\right)(T-t)\right],$$

where all exposure-dependent terms have been collected in

$$q(a_0):=\gamma\lambda(1-a_0)+\frac{\gamma^2\sigma^2}{2}a_0^2.$$

The first term represents the utility-adjusted cost of reinsurance, while the second term represents the utility cost of retained risk. Maximizing expected exponential utility is equivalent to minimizing $q(a_0)$. Since

$$q'(a_0)=-\gamma\lambda+\gamma^2\sigma^2a_0,$$

the unconstrained minimizer is

$$a_0=\frac{\lambda}{\gamma\sigma^2}.$$

Taking the constraint $a_0\in[0,1]$ into account gives

$$\boxed{\bar a_0=\min\left\{\frac{\lambda}{\gamma\sigma^2},1\right\}.}$$

### Step 2: First fix a deterministic default date

To construct a candidate strategy, we first consider a simpler auxiliary problem: fix a deterministic date $u\in[0,T]$ and suppose that default occurs exactly at $u$.

The exposure is then

$$a(t)=\begin{cases}a_1(t), & t<u,\\ \bar a_0, & t\geq u.\end{cases}$$

For a deterministic exposure function $a_1$, terminal surplus becomes

$$\begin{aligned}
R^{a,u}(T)
={}&x+\int_0^u\left[\mu-(1-a_1(s))\lambda\right]\mathrm{d}s+\int_u^T\left[\mu-(1-\bar a_0)\lambda\right]\mathrm{d}s\\
&+\sigma\int_0^u a_1(s)\,\mathrm{d}W_s+\sigma\bar a_0(W_T-W_u)-C-(1-a_1(u))F.
\end{aligned}$$

Its mean is

$$m_a(u)=x+\mu T-\lambda\int_0^u(1-a_1(s))\,\mathrm{d}s-\lambda(1-\bar a_0)(T-u)-C-(1-a_1(u))F,$$

and its variance is

$$v_a(u)=\sigma^2\left[\int_0^u a_1(s)^2\,\mathrm{d}s+\bar a_0^2(T-u)\right].$$

Since $R^{a,u}(T)$ is normally distributed,

$$J(a,u):=\mathbb{E}\left[-\mathrm{e}^{-\gamma R^{a,u}(T)}\right]=-\exp\left[\Psi_a(u)\right],$$

where

$$\Psi_a(u):=-\gamma m_a(u)+\frac{\gamma^2}{2}v_a(u).$$

Assuming $\tau$ is an independent random default time, one has

$$\mathbb{E}\left[-\mathrm{e}^{-\gamma R^{a,\tau}(T)}\right]=\mathbb{E}\left[-\mathrm{e}^{\Psi_a(\tau)}\right],$$

### Step 3: Construct a default-date-indifferent strategy

Since $-\mathrm{e}^z$ is decreasing in $z$, an unfavorable default date is one that makes $\Psi_a(u)$ large.

Thus, we construct an equalizer strategy for which every deterministic default date gives the same value:

$$\frac{\mathrm{d}}{\mathrm{d}u}\Psi_a(u) = 0.$$

Differentiating the mean gives

$$\begin{aligned}
m_a'(u)
&=-\lambda(1-a_1(u))+\lambda(1-\bar a_0)+Fa_1'(u)\\
&=\lambda(a_1(u)-\bar a_0)+Fa_1'(u).
\end{aligned}$$

Differentiating the variance gives

$$v_a'(u)=\sigma^2\left[a_1(u)^2-\bar a_0^2\right].$$

Consequently,

$$\Psi_a'(u)=-\gamma\lambda(a_1(u)-\bar a_0)-\gamma Fa_1'(u)+\frac{\gamma^2\sigma^2}{2}\left[a_1(u)^2-\bar a_0^2\right].$$

Setting $\Psi_a'(u)=0$ yields

$$Fa_1'(u)=(a_1(u)-\bar a_0)\left[-\lambda+\frac{\gamma\sigma^2}{2}(a_1(u)+\bar a_0)\right].$$

### Step 4: Insert the optimal post-default exposure

Suppose that $\lambda<\gamma\sigma^2$. Then the post-default optimum is interior and

$$\bar a_0=\frac{\lambda}{\gamma\sigma^2}, \qquad \lambda=\gamma\sigma^2\bar a_0.$$

Therefore,

$$\begin{aligned}
-\lambda+\frac{\gamma\sigma^2}{2}(a_1(u)+\bar a_0)
&=-\gamma\sigma^2\bar a_0+\frac{\gamma\sigma^2}{2}(a_1(u)+\bar a_0)\\
&=\frac{\gamma\sigma^2}{2}(a_1(u)-\bar a_0).
\end{aligned}$$

The default-date-indifference condition thus simplifies to

$$\boxed{a_1'(u)=\frac{\gamma\sigma^2}{2F}(a_1(u)-\bar a_0)^2.}$$

Since the right-hand side is nonnegative, the retained exposure increases as maturity approaches.

### Step 5: Determine the terminal condition

Suppose default occurs exactly at $T$. At this time, the exposure no longer affects future premium payments, diffusion risk, or surplus accumulation. It only affects the immediate default loss

$$C+(1-a_1(T))F.$$

This loss is minimized by choosing the largest admissible retained exposure. Hence,

$$\boxed{a_1(T)=1.}$$

Economically, there is no reason to purchase reinsurance at the final instant: it provides no remaining risk protection but still exposes the insurer to the reinsurer's default.

### Step 6: Solve the differential equation

We solve

$$a_1'(t)=\frac{\gamma\sigma^2}{2F}(a_1(t)-\bar a_0)^2, \qquad a_1(T)=1.$$

Separating variables and integrating from $t$ to $T$ gives

$$\frac{1}{a_1(t)-\bar a_0}-\frac{1}{1-\bar a_0}=\frac{\gamma\sigma^2}{2F}(T-t).$$

Therefore,

$$\frac{1}{a_1(t)-\bar a_0}=\frac{\gamma\sigma^2}{2F}(T-t)+\frac{1}{1-\bar a_0},$$

and hence

$$\boxed{\bar a_1(t)=\bar a_0+\frac{2F}{\gamma\sigma^2(T-t)+\frac{2F}{1-\bar a_0}}.}$$

If $\lambda\geq\gamma\sigma^2$, then $\bar a_0=1$ and the optimal pre-default exposure is also

$$\boxed{\bar a_1(t)=1.}$$

To see this consider the following:
The constant function
$$\bar a_1(t) \equiv 1$$
clearly solves this terminal-value problem. Moreover, the right-hand side is continuously differentiable in $\bar a_1$, hence locally Lipschitz. Therefore, the solution satisfying $\bar a_1(T) = 1$ is unique, and it follows that
$$\boxed{\bar a_1(t)=1\quad\text{for all }t\in[0,T].}$$

### Interpretation

- Early in the contract, the insurer buys more reinsurance because many years of diffusion risk remain.
- Close to maturity, retaining risk becomes more attractive because little future volatility remains, while default-related replacement costs still matter.
- A larger $F$ makes reinsurer default more costly and slows the transition toward full retention.
- Greater risk aversion or volatility reduces retained exposure.
- When $\lambda\geq\gamma\sigma^2$, reinsurance is too expensive relative to its risk reduction and full retention is optimal.

## Exercise 3 (Bonus)

Perform an internet search for current dynamic models for the evolution of the average temperature increase in the last ten years.

### Historical global temperature anomalies

We now leave the portfolio setting and consider a purely environmental application. The bundled data contain annual global HadCRUT5 temperature anomalies from 1850 through 2020. The original observations use a different reference period, so we rebase them to 1850--1900:

$$K_y=T_y-\frac{1}{51}\sum_{s=1850}^{1900}T_s.$$

Rebasing shifts the complete series vertically. It does not change annual increments, the estimated drift, or the estimated volatility.

For annual observations, the arithmetic-Brownian model is

$$K_{y+1}=K_y+\mu+\sigma Z_{y+1}, \qquad Z_{y+1}\sim\mathcal{N}(0,1).$$

We estimate $\mu$ and $\sigma$ from the historical annual changes $\Delta K_y=K_y-K_{y-1}$. Arithmetic rather than geometric Brownian motion is used because temperature anomalies may be negative and percentage changes have no useful physical interpretation here.

In [ ]:
temperature_data = pd.read_csv(
    "data/global_temperature_anomaly_1850_2020.csv"
)

history_years = temperature_data["year"].to_numpy()
raw_anomaly   = temperature_data["anomaly_c"].to_numpy()

history_anomaly = rebase_temperature_anomaly(history_years, raw_anomaly, baseline_start=1850, baseline_end=1900)

historical_drift, historical_volatility = (
    estimate_arithmetic_brownian_parameters(
        history_years,
        history_anomaly,
    )
)

print(f"Historical annual drift:      {historical_drift:.4f} °C/year")
print(f"Historical annual volatility: {historical_volatility:.4f} °C/sqrt(year)")

check_value(
    "Mean anomaly over the 1850--1900 baseline",
    np.mean(history_anomaly[(history_years >= 1850) & (history_years <= 1900)]),
    0.0,
    abs_tol=1e-12,
)
check_condition(
    "Historical sample is 1850--2020",
    history_years[0] == 1850 and history_years[-1] == 2020,
)

### Conditional IPCC scenario paths

The IPCC AR6 reports central warming estimates relative to 1850--1900 for three twenty-year periods. We represent these periods by their midpoints 2030, 2050, and 2090:

| Scenario | 2021-2040 | 2041-2060 | 2081-2100 |
| --- | ---: | ---: | ---: |
| SSP1-1.9 | $1.5^\circ$ C | $1.6^\circ$ C | $1.4^\circ$ C |
| SSP1-2.6 | $1.5^\circ$ C | $1.7^\circ$ C | $1.8^\circ$ C |
| SSP2-4.5 | $1.5^\circ$ C | $2.0^\circ$ C | $2.7^\circ$ C |
| SSP5-8.5 | $1.6^\circ$ C | $2.4^\circ$ C | $4.4^\circ$ C |

Linear interpolation between these anchors defines a conditional mean $m_s(y)$; the 2090 level is held constant through 2100. The scenario-dependent arithmetic-Brownian simulation is

$$K_{y+1}^{(s)}=K_y^{(s)}+\left[m_s(y+1)-m_s(y)\right]+\widehat{\sigma}Z_{y+1}.$$

Note that the IPCC values are not literal annual drift parameters. The bracketed term converts period-level scenario information into a simple annual mean increment for this exercise. The SSPs are conditional scenarios; we do not assign probabilities to the scenarios themselves.

In [ ]:
def simulate_climate_scenario(initial_anomaly: float, forecast_years: np.ndarray, scenario_mean: np.ndarray, annual_volatility: float, num_paths: int = 5_000) -> SDEOutput:
    '''Simulate annual temperature anomalies around a conditional scenario mean.'''

    forecast_years = np.asarray(forecast_years)
    scenario_mean  = np.asarray(scenario_mean)

    if forecast_years.shape != scenario_mean.shape:
        raise ValueError("Forecast years and scenario mean must have the same shape.")
    if not np.allclose(np.diff(forecast_years), 1):
        raise ValueError("The forecast grid must be annual.")
    if annual_volatility < 0:
        raise ValueError("Annual volatility must be non-negative.")
    if not np.isclose(scenario_mean[0], initial_anomaly):
        raise ValueError("The scenario mean must start at the observed 2020 anomaly.")

    rng = np.random.default_rng(42)

    mean_increments = np.diff(scenario_mean)
    random_increments = rng.normal(
        scale=annual_volatility,
        size=(len(forecast_years) - 1, num_paths),
    )

    paths = np.zeros((len(forecast_years), num_paths))
    paths[0, :] = initial_anomaly
    paths[1:, :] = initial_anomaly + np.cumsum(
        mean_increments[:, np.newaxis] + random_increments,
        axis=0,
    )

    return SDEOutput(
        time_grid=forecast_years,
        paths={
            "temperature_anomaly": paths,
            "scenario_mean": scenario_mean,
        },
    )

### Numerical validation

For a fixed scenario, the Monte Carlo mean should follow the prescribed conditional mean. After subtracting the scenario increment, the standard deviation of the simulated one-year innovations should reproduce the historical volatility estimate.

In [ ]:
forecast_years = np.arange(2020, 2101)
scenario_mean = build_climate_scenario_mean(
    forecast_years,
    history_anomaly[-1],
    IPCC_AR6_SCENARIO_CENTRAL["SSP2-4.5"],
)

climate_paths = simulate_climate_scenario(
    initial_anomaly=history_anomaly[-1],
    forecast_years=forecast_years,
    scenario_mean=scenario_mean,
    annual_volatility=historical_volatility,
    num_paths=25_000,
)

simulated_innovations = (
    np.diff(climate_paths.paths["temperature_anomaly"], axis=0)
    - np.diff(scenario_mean)[:, np.newaxis]
)

terminal_standard_error = (
    historical_volatility
    * np.sqrt(forecast_years[-1] - forecast_years[0])
    / np.sqrt(25_000)
)

check_value(
    "Mean simulated anomaly in 2100",
    np.mean(climate_paths.paths["temperature_anomaly"][-1, :]),
    scenario_mean[-1],
    abs_tol=4 * terminal_standard_error,
)
check_value(
    "Standard deviation of annual innovations",
    np.std(simulated_innovations),
    historical_volatility,
    abs_tol=1e-3,
)

### Interactive scenario experiment

The first panel joins the 1850-2020 observations to all four conditional scenario means. The second shows a simulation fan around the selected scenario. The third estimates the probability that an individual simulated year exceeds a chosen threshold. This is not the probability that an IPCC scenario occurs, and a single annual exceedance is not the same as a twenty-year average warming level.

In [ ]:
explore_climate_scenarios(history_years, history_anomaly, historical_volatility, simulate_climate_scenario)

### Interpretation and limitations

- Historical annual changes identify short-run variability, while the IPCC values determine the conditional future mean. Keeping these two roles separate makes the assumptions visible.
- Low-emissions and high-emissions scenarios are similar in the near term but diverge strongly later in the century.
- The arithmetic-Brownian model permits temporary cooling as well as warming and makes uncertainty grow with the square root of time.
- The volatility multiplier is a sensitivity analysis, not a fitted climate parameter.
- Brownian increments are independent and the estimated volatility is treated as constant. Real temperatures exhibit persistence, forcing dynamics, measurement uncertainty, and richer physical dependence. The model is therefore a heavily simplified illustration rather than a climate forecast.
- The historical data are HadCRUT5 annual global anomalies distributed by the Met Office Hadley Centre and accessed through Our World in Data. The scenario anchors are the central estimates in IPCC AR6 WGI Table SPM.1.

### Data sources

- [Met Office HadCRUT5 data and documentation](https://www.metoffice.gov.uk/hadobs/hadcrut5/)
- [Our World in Data: temperature change relative to the pre-industrial period](https://ourworldindata.org/grapher/temperature-anomaly)
- [IPCC AR6 Working Group I Summary for Policymakers](https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_SPM_final.pdf)